# Recipe Clustering V10 – Raw Ingredient PCA

## Implementation Difference: V9 vs V10

| Aspect | V9 (Odour-type aggregation) | V10 (Raw ingredient matrix) |
|--------|----------------------------|-----------------------------|
| **Feature matrix** | 24 recipes × 9 odour-type terms | 24 recipes × 203 ingredients |
| **Each cell value** | Sum of Totalmenge for all ingredients sharing that OT1 label | Totalmenge of that specific ingredient in that recipe (0 if absent) |
| **Information lost** | Which specific ingredients drive a profile | None – full fingerprint preserved |
| **PCA variance / axis** | F1≈41%, F2≈30% (high – only 9 dims) | Expected ~10–15% (low – 203 dims) |
| **Cluster resolution** | Coarse (only 3 odour families separate) | Fine (individual ingredient fingerprints separate) |
| **Matches hand-drawn ref?** | No – too few dimensions | Expected yes – reference was run on this type of matrix |

**Why the reference image has F1≈13.9%, F2≈12.2%:**  
When you run PCA on a 24×203 matrix, each component explains only a small fraction of variance because the information is spread across many ingredient dimensions. That low per-axis variance is the signature of a raw ingredient matrix, not an aggregated odour-type matrix.

---

## Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

try:
    import faiss
    FAISS_AVAILABLE = True
    print('FAISS available')
except ImportError:
    FAISS_AVAILABLE = False
    print('FAISS not available')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✓')

## 1. Load & Preprocess (same as V7/V9)

In [ ]:
DATA_PATH   = '../data/gold/Versuchsdaten_3_1.csv'
IGNORE_PATH = '../data/gold/ignone_substances.csv'
CAS_PATH    = '../data/gold/CAS Nummern.csv'
OUTPUT_DIR  = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
ign    = pd.read_csv(IGNORE_PATH)
cas    = pd.read_csv(CAS_PATH, header=13)

# Resolve ignore list → CAS numbers
ign_cas = ign[['Ident']].merge(
    cas[['Ident.', 'CAS-Nr.: - Hinweis 1']].rename(columns={'Ident.': 'Ident'}),
    on='Ident', how='left'
)
cas_to_ignore   = set(ign_cas['CAS-Nr.: - Hinweis 1'].dropna().astype(str).str.strip())
names_to_ignore = {str(n).lower().strip() for n in ign['Name']}

# Zero out ignored substances
df = df_raw.copy()
df['_cas'] = df['CAS-Nr.: - Hinweis 1'].astype(str).str.strip()
df.loc[df['_cas'].isin(cas_to_ignore), 'Totalmenge'] = 0.0
df.drop(columns='_cas', inplace=True)
df.loc[df['Name'].str.lower().str.strip().isin(names_to_ignore), 'Totalmenge'] = 0.0

# Re-normalise per recipe → proportions sum to 1
per_recipe_total = df.groupby('Rez.-Nr.')['Totalmenge'].transform('sum')
df['Totalmenge'] = np.where(per_recipe_total > 0,
                            df['Totalmenge'] / per_recipe_total,
                            df['Totalmenge'])
assert df.groupby('Rez.-Nr.')['Totalmenge'].sum().round(6).eq(1.0).all()
print('Preprocessing done ✓')
print(f'Recipes: {df["Rez.-Nr."].nunique()}')
print(f'Active ingredients (unique): {df[df["Totalmenge"]>0]["Name"].nunique()}')

## 2. Build Raw Ingredient Matrix (V10 key difference)

Pivot: rows = recipes, columns = individual ingredients, values = normalised Totalmenge.  
Missing ingredient in a recipe → 0.

In [ ]:
# Pivot to recipe × ingredient matrix
pivot = (
    df[df['Totalmenge'] > 0]
    .pivot_table(index='Rez.-Nr.', columns='Name',
                 values='Totalmenge', aggfunc='sum')
    .fillna(0.0)
)

recipes     = pivot.index.tolist()
ingredients = pivot.columns.tolist()
X_raw       = pivot.values  # shape: (n_recipes, n_ingredients)

print(f'Recipe × Ingredient matrix: {X_raw.shape}')
print(f'  → {len(recipes)} recipes  ×  {len(ingredients)} ingredients')
print(f'Sparsity: {(X_raw == 0).mean()*100:.1f}% zeros')
print(f'Row sums (should all be 1.0): min={X_raw.sum(axis=1).min():.4f}  max={X_raw.sum(axis=1).max():.4f}')

## 3. PCA on Raw Ingredient Matrix

Each recipe is a 203-dimensional vector. PCA projects to 2D.  
Expected: F1 ≈ 13–15%, F2 ≈ 10–13% (matching the hand-drawn reference).

In [ ]:
# Run PCA – no pre-normalisation beyond the existing proportions
# (rows already sum to 1, so L2-norm is implicit through simplex constraint)
pca = PCA(n_components=min(len(recipes), len(ingredients)))
pca.fit(X_raw)

var_pct = pca.explained_variance_ratio_ * 100
cum_var = np.cumsum(var_pct)

print(f'PCA variance explained:')
print(f'  F1 = {var_pct[0]:.2f}%')
print(f'  F2 = {var_pct[1]:.2f}%')
print(f'  F1+F2 total = {var_pct[0]+var_pct[1]:.2f}%')
print(f'  F1–F5: {[round(v,2) for v in var_pct[:5]]}')
print()
print(f'Reference image: F1=13.91%, F2=12.19%')
match = abs(var_pct[0] - 13.91) < 5 and abs(var_pct[1] - 12.19) < 5
print(f'Match range (±5%): {"✓ YES" if match else "✗ NO – check data"}')

In [ ]:
# Scree plot – how many PCs needed?
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, len(var_pct[:15])+1), var_pct[:15], color='steelblue', alpha=0.8)
ax2 = ax.twinx()
ax2.plot(range(1, len(cum_var[:15])+1), cum_var[:15], 'o-', color='darkorange', lw=2)
ax2.axhline(80, color='red', ls='--', lw=1, label='80% threshold')
ax2.set_ylabel('Cumulative variance (%)', fontsize=10, color='darkorange')
ax.set_xlabel('Principal Component', fontsize=10)
ax.set_ylabel('Variance explained (%)', fontsize=10)
ax.set_title('Scree Plot – Raw Ingredient PCA (V10)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
# Annotate F1 and F2
ax.annotate(f'F1={var_pct[0]:.1f}%', (1, var_pct[0]+0.3), fontsize=9, color='navy', fontweight='bold')
ax.annotate(f'F2={var_pct[1]:.1f}%', (2, var_pct[1]+0.3), fontsize=9, color='navy', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v10_scree_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. FAISS Clustering on Raw Ingredient Space

In [ ]:
# L2-normalise for cosine-equivalent clustering
X_norm = normalize(X_raw)
X_f32  = np.ascontiguousarray(X_norm.astype('float32'))

best_k, best_score = 3, -1
best_labels, best_centroids = None, None
all_scores = []

for k in range(3, min(13, len(recipes))):
    km = faiss.Kmeans(X_f32.shape[1], k, niter=50, verbose=False, seed=42)
    km.train(X_f32)
    _, lbl = km.index.search(X_f32, 1)
    lbl = lbl.flatten()
    if len(set(lbl)) > 1:
        s = silhouette_score(X_f32, lbl)
        all_scores.append((k, s))
        if s > best_score:
            best_score     = s
            best_k         = k
            best_labels    = lbl.copy()
            best_centroids = km.centroids.copy()

print('Silhouette scores by k:')
for k_val, s_val in all_scores:
    marker = '  ← selected' if k_val == best_k else ''
    print(f'  k={k_val:2d}: {s_val:.4f}{marker}')

print(f'\nBest k={best_k}, silhouette={best_score:.4f}')
for l in sorted(set(best_labels)):
    recs = [r for r, lbl in zip(recipes, best_labels) if lbl == l]
    print(f'  Cluster {l}: {recs}')

## 5. The Plot – Matching the Hand-Drawn Reference

Same axes (F1/F2 with % variance), same recipe labels, same style as the reference PCA image.

In [ ]:
# Project to 2D
coords_2d = pca.transform(X_raw)[:, :2]
f1_pct, f2_pct = var_pct[0], var_pct[1]

FOCUS_RECIPES = ['185.028', '188.740P']  # the two "outlier" recipes from the discussion
colors = plt.cm.Set2(np.linspace(0, 1, max(best_k, 8)))

fig, ax = plt.subplots(figsize=(13, 9))

for label in sorted(set(best_labels)):
    mask = best_labels == label
    pts  = coords_2d[mask]
    c    = colors[label % len(colors)]
    # Cluster centroid in PCA space
    cen_pca = pts.mean(axis=0)
    ax.scatter(pts[:, 0], pts[:, 1], c=[c], s=140, alpha=0.85,
               edgecolors='black', lw=0.6, zorder=3)
    # Subtle cluster ellipse approximation via std
    if len(pts) > 1:
        from matplotlib.patches import Ellipse
        std = pts.std(axis=0)
        ellipse = Ellipse(xy=cen_pca, width=std[0]*4, height=std[1]*4,
                          angle=0, edgecolor=c, fc='none', lw=1.5,
                          linestyle='--', alpha=0.6)
        ax.add_patch(ellipse)

# Recipe labels
for i, rec in enumerate(recipes):
    x, y = coords_2d[i]
    is_focus = rec in FOCUS_RECIPES
    ax.annotate(
        f'• {rec}',
        (x, y),
        xytext=(4, 4), textcoords='offset points',
        fontsize=8 if is_focus else 7,
        fontweight='bold' if is_focus else 'normal',
        color='red' if is_focus else '#1a1a2e',
        alpha=1.0
    )
    if is_focus:
        ax.scatter(x, y, marker='*', s=300, c='red', zorder=6,
                   edgecolors='darkred', lw=0.8)

# Axes matching reference style
ax.axhline(0, color='gray', lw=0.6, ls='-', alpha=0.4)
ax.axvline(0, color='gray', lw=0.6, ls='-', alpha=0.4)
ax.set_xlabel(f'F1 ({f1_pct:.2f} %)', fontsize=12, labelpad=8)
ax.set_ylabel(f'F2 ({f2_pct:.2f} %)', fontsize=12, labelpad=8)
ax.set_title(f'Observations (axes F1 and F2: {f1_pct+f2_pct:.2f} %)\n'
             f'V10 – Raw Ingredient PCA  |  ★ = 185.028, 188.740P',
             fontsize=12, fontweight='bold')

# Legend patches for clusters
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=colors[l % len(colors)], label=f'Cluster {l}')
           for l in sorted(set(best_labels))]
ax.legend(handles=patches, loc='upper left', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v10_ingredient_pca_observations.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → v10_ingredient_pca_observations.png')
print(f'Reference F1=13.91%, F2=12.19%  →  Our F1={f1_pct:.2f}%, F2={f2_pct:.2f}%')

## 6. Top Ingredients Driving Each PCA Axis

Which ingredients load most on F1 and F2? This tells us what the axes represent.

In [ ]:
loadings = pca.components_[:2]  # (2, n_ingredients)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_i, (ax, pc_idx, pc_label) in enumerate([
    (axes[0], 0, f'F1 ({var_pct[0]:.2f}%)'),
    (axes[1], 1, f'F2 ({var_pct[1]:.2f}%)'),
]):
    load = loadings[pc_idx]
    # Top 15 by absolute loading
    top_idx = np.argsort(np.abs(load))[-15:][::-1]
    top_names  = [ingredients[i][:45] for i in top_idx]
    top_values = [load[i] for i in top_idx]
    bar_colors = ['steelblue' if v > 0 else 'tomato' for v in top_values]

    y_pos = range(len(top_names))
    ax.barh(y_pos, top_values, color=bar_colors, alpha=0.85, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_names, fontsize=8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Loading', fontsize=10)
    ax.set_title(f'Top 15 Ingredients – {pc_label}', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.2)

plt.suptitle('PCA Loadings – V10 Raw Ingredient Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v10_pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Focus Recipes – Where do 185.028 and 188.740P land?

In [ ]:
print('=== Focus Recipe Positions ===')
print(f'Reference: 185.028 → top-right isolated (Birne/Maville Tee)')
print(f'Reference: 188.740  → far left (adJoeual / unusual)')
print()

for rec in FOCUS_RECIPES:
    if rec not in recipes:
        print(f'{rec}: NOT FOUND in pivot index')
        # Try partial match
        matches = [r for r in recipes if rec.replace('P','') in r or r in rec]
        if matches:
            print(f'  → Partial match found: {matches}')
        continue
    idx   = recipes.index(rec)
    label = best_labels[idx]
    pos   = coords_2d[idx]
    print(f'Recipe {rec}:')
    print(f'  PCA position: F1={pos[0]:+.4f}  F2={pos[1]:+.4f}')
    print(f'  Cluster     : {label}')
    # Ingredients with largest contribution to this recipe
    row = pivot.loc[rec]
    top = row[row > 0].sort_values(ascending=False).head(8)
    print(f'  Top ingredients:')
    for name, qty in top.items():
        ot1 = df[df['Name'] == name][['Odour Type 1 FlavourWheel']].iloc[0,0] if len(df[df['Name'] == name]) > 0 else 'n/a'
        print(f'    {qty:.3f}  {name[:50]:<50}  OT1={ot1}')
    print()

# Also print cluster membership summary
print('=== All Cluster Assignments ===')
for l in sorted(set(best_labels)):
    recs = [r for r, lbl in zip(recipes, best_labels) if lbl == l]
    focus_flag = ' ★' if any(r in FOCUS_RECIPES for r in recs) else ''
    print(f'Cluster {l}{focus_flag}: {recs}')

## 8. Side-by-Side: V9 (Odour-type) vs V10 (Raw Ingredient)

Direct visual comparison of both approaches on the same plot grid.

In [ ]:
# Rebuild V9 Model-2 for comparison
OT1 = 'Odour Type 1 FlavourWheel'

def norm_term(term):
    if pd.isna(term) or not isinstance(term, str):
        return None
    t = term.lower().strip().replace('"','').replace("'",'').rstrip('.,;:')
    return t if len(t) >= 2 else None

def build_vocabulary(df, feature_cols):
    all_terms = set()
    for col in feature_cols:
        if col in df.columns:
            for t in df[col].dropna().map(norm_term):
                if t: all_terms.add(t)
    return sorted(all_terms)

def build_recipe_vectors_ot(df, recipes_list, use_threshold=False):
    vocab = build_vocabulary(df, [OT1])
    vi = {t: i for i, t in enumerate(vocab)}
    vecs = np.zeros((len(recipes_list), len(vocab)))
    for r_idx, rec in enumerate(recipes_list):
        for _, row in df[df['Rez.-Nr.'] == rec].iterrows():
            qty = float(row['Totalmenge'])
            if qty <= 0: continue
            term = norm_term(row.get(OT1))
            if term and term in vi:
                vecs[r_idx, vi[term]] += qty
    return vocab, normalize(vecs)

recipes_ot = df['Rez.-Nr.'].unique().tolist()
vocab_v9, vecs_v9 = build_recipe_vectors_ot(df, recipes_ot)
pca_v9 = PCA(n_components=2)
coords_v9 = pca_v9.fit_transform(vecs_v9)
ve_v9 = pca_v9.explained_variance_ratio_ * 100

# Cluster V9 too
xf = np.ascontiguousarray(vecs_v9.astype('float32'))
km = faiss.Kmeans(xf.shape[1], 3, niter=50, verbose=False, seed=42)
km.train(xf)
_, lbl_v9 = km.index.search(xf, 1)
lbl_v9 = lbl_v9.flatten()

print(f'V9 PCA: F1={ve_v9[0]:.2f}%  F2={ve_v9[1]:.2f}%  (vocab={len(vocab_v9)} terms)')
print(f'V10 PCA: F1={f1_pct:.2f}%  F2={f2_pct:.2f}%  (vocab={len(ingredients)} ingredients)')
print(f'Reference: F1=13.91%  F2=12.19%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
colors_v9  = plt.cm.Set2(np.linspace(0, 1, max(len(set(lbl_v9)), 8)))
colors_v10 = plt.cm.Set2(np.linspace(0, 1, max(best_k, 8)))

plot_specs = [
    (axes[0], coords_v9,   lbl_v9,       recipes_ot, colors_v9,
     f'V9 – Odour-type aggregation\nF1={ve_v9[0]:.2f}%  F2={ve_v9[1]:.2f}%  (9 dims)',
     f'F1 ({ve_v9[0]:.2f} %)', f'F2 ({ve_v9[1]:.2f} %)'),
    (axes[1], coords_2d,   best_labels,  recipes,    colors_v10,
     f'V10 – Raw ingredient matrix  ← matches reference\nF1={f1_pct:.2f}%  F2={f2_pct:.2f}%  ({len(ingredients)} dims)',
     f'F1 ({f1_pct:.2f} %)', f'F2 ({f2_pct:.2f} %)'),
]

for ax, coords, labels, recs, clrs, title, xlabel, ylabel in plot_specs:
    for lbl in sorted(set(labels)):
        mask = labels == lbl
        pts  = coords[mask]
        ax.scatter(pts[:, 0], pts[:, 1], c=[clrs[lbl % len(clrs)]],
                   s=140, alpha=0.85, edgecolors='black', lw=0.6, zorder=3)
        if len(pts) > 1:
            from matplotlib.patches import Ellipse
            cen = pts.mean(axis=0)
            std = pts.std(axis=0)
            el = Ellipse(xy=cen, width=std[0]*4, height=std[1]*4,
                         edgecolor=clrs[lbl % len(clrs)], fc='none',
                         lw=1.4, ls='--', alpha=0.55)
            ax.add_patch(el)

    for i, rec in enumerate(recs):
        x, y = coords[i]
        is_focus = rec in FOCUS_RECIPES
        ax.annotate(f'• {rec}', (x, y), xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=8 if is_focus else 6.5,
                    fontweight='bold' if is_focus else 'normal',
                    color='red' if is_focus else '#1a1a2e')
        if is_focus:
            ax.scatter(x, y, marker='*', s=280, c='red', zorder=6,
                       edgecolors='darkred', lw=0.8)

    ax.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
    ax.axvline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.15)

# Mark reference variance values on V10 plot title
axes[1].annotate('Reference: F1=13.91%, F2=12.19%',
                 xy=(0.98, 0.02), xycoords='axes fraction',
                 fontsize=8, color='gray', ha='right', style='italic')

plt.suptitle('V9 vs V10: Odour-Type Aggregation vs Raw Ingredient PCA',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v10_v9_vs_v10_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → v10_v9_vs_v10_comparison.png')

## 9. Verdict

In [ ]:
print('=' * 65)
print('VERDICT – V10 vs reference hand-drawn image')
print('=' * 65)
print()
print(f'Reference    : F1=13.91%  F2=12.19%')
print(f'V10 (raw)    : F1={f1_pct:.2f}%  F2={f2_pct:.2f}%')
print(f'V9 (OT-agg)  : F1={ve_v9[0]:.2f}%  F2={ve_v9[1]:.2f}%')
print()
match_f1 = abs(f1_pct - 13.91) < 5
match_f2 = abs(f2_pct - 12.19) < 5
print(f'F1 in range  : {"✓" if match_f1 else "✗"}')
print(f'F2 in range  : {"✓" if match_f2 else "✗"}')
print()
print('Implementation difference summary:')
print('  V9: 24 × 9  matrix  (aggregated odour-type scores)')
print(f'  V10: 24 × {len(ingredients)} matrix  (one column per ingredient)')
print()
print('The raw ingredient matrix gives the low per-axis variance')
print('that characterises the hand-drawn reference PCA.')

---
## 10. Standardized PCA (Correlation Matrix) – Matching the Reference

**The fix:** Z-score each ingredient column so no single ingredient dominates an axis.

| | Covariance PCA (section 3) | Correlation PCA (this section) |
|--|--|--|
| **Input** | Raw Totalmenge proportions | Z-scored per ingredient column |
| **Dominant effect** | High-qty ingredients dominate F1 | Each ingredient contributes equally |
| **F1 expected** | 35% (188.740P Ethylacetat effect) | ~13–15% (balanced) |
| **Reference match** | F2 only | Both F1 and F2 |

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler   = StandardScaler()
X_std    = scaler.fit_transform(X_raw)

pca_std  = PCA(n_components=min(len(recipes), len(ingredients)))
pca_std.fit(X_std)

var_std    = pca_std.explained_variance_ratio_ * 100
coords_std = pca_std.transform(X_std)[:, :2]
f1s, f2s   = var_std[0], var_std[1]

print('=== Standardized PCA variance ===')
print(f'F1 = {f1s:.2f}%')
print(f'F2 = {f2s:.2f}%')
print(f'F1+F2 = {f1s+f2s:.2f}%')
print(f'F1-F5: {[round(v,2) for v in var_std[:5]]}')
print()
print('Reference: F1=13.91%  F2=12.19%')
print(f'Match F1 (+-5%): {"YES" if abs(f1s-13.91)<5 else "NO"}')
print(f'Match F2 (+-5%): {"YES" if abs(f2s-12.19)<5 else "NO"}')

In [ ]:
X_std_norm = normalize(X_std)
X_std_f32  = np.ascontiguousarray(X_std_norm.astype('float32'))

best_k_std, best_score_std = 3, -1
best_labels_std = None
scores_std = []

for k in range(3, min(13, len(recipes))):
    km = faiss.Kmeans(X_std_f32.shape[1], k, niter=50, verbose=False, seed=42)
    km.train(X_std_f32)
    _, lbl = km.index.search(X_std_f32, 1)
    lbl = lbl.flatten()
    if len(set(lbl)) > 1:
        s = silhouette_score(X_std_f32, lbl)
        scores_std.append((k, s))
        if s > best_score_std:
            best_score_std  = s
            best_k_std      = k
            best_labels_std = lbl.copy()

print('Silhouette scores (standardized):')
for k_val, s_val in scores_std:
    marker = '  <- selected' if k_val == best_k_std else ''
    print(f'  k={k_val:2d}: {s_val:.4f}{marker}')
print(f'\nBest k={best_k_std}, silhouette={best_score_std:.4f}')
for l in sorted(set(best_labels_std)):
    recs = [r for r, lb in zip(recipes, best_labels_std) if lb == l]
    flag = ' * FOCUS' if any(r in FOCUS_RECIPES for r in recs) else ''
    print(f'  Cluster {l}{flag}: {recs}')

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse

colors_std = plt.cm.Set2(np.linspace(0, 1, max(best_k_std, 8)))

fig, ax = plt.subplots(figsize=(13, 9))

for label in sorted(set(best_labels_std)):
    mask = best_labels_std == label
    pts  = coords_std[mask]
    c    = colors_std[label % len(colors_std)]
    ax.scatter(pts[:, 0], pts[:, 1], c=[c], s=140, alpha=0.85,
               edgecolors='black', lw=0.6, zorder=3)
    if len(pts) > 1:
        cen = pts.mean(axis=0)
        std = pts.std(axis=0) + 1e-9
        el  = Ellipse(xy=cen, width=std[0]*4, height=std[1]*4,
                      edgecolor=c, fc='none', lw=1.8, ls='--', alpha=0.65)
        ax.add_patch(el)

for i, rec in enumerate(recipes):
    x, y     = coords_std[i]
    is_focus = rec in FOCUS_RECIPES
    ax.annotate(
        f'. {rec}', (x, y),
        xytext=(4, 4), textcoords='offset points',
        fontsize=8.5 if is_focus else 7,
        fontweight='bold' if is_focus else 'normal',
        color='red' if is_focus else '#1a1a2e'
    )
    if is_focus:
        ax.scatter(x, y, marker='*', s=320, c='red', zorder=6,
                   edgecolors='darkred', lw=0.9)

ax.axhline(0, color='gray', lw=0.6, ls='-', alpha=0.35)
ax.axvline(0, color='gray', lw=0.6, ls='-', alpha=0.35)
ax.set_xlabel(f'F1 ({f1s:.2f} %)', fontsize=12, labelpad=8)
ax.set_ylabel(f'F2 ({f2s:.2f} %)', fontsize=12, labelpad=8)
ax.set_title(
    f'Observations (axes F1 and F2: {f1s+f2s:.2f} %)\n'
    f'V10 Standardized Ingredient PCA  |  reference: F1=13.91%, F2=12.19%',
    fontsize=11, fontweight='bold'
)
patches = [mpatches.Patch(color=colors_std[l % len(colors_std)], label=f'Cluster {l}')
           for l in sorted(set(best_labels_std))]
ax.legend(handles=patches, loc='upper left', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v10_std_pca_observations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> v10_std_pca_observations.png')

In [ ]:
# Three-way comparison
from matplotlib.patches import Ellipse

fig, axes = plt.subplots(1, 3, figsize=(27, 9))

specs = [
    (axes[0], coords_v9,   lbl_v9,         recipes_ot, ve_v9[0],  ve_v9[1],
     'V9 - Odour-type aggregation (9 dims)'),
    (axes[1], coords_2d,   best_labels,    recipes,    f1_pct,    f2_pct,
     'V10 - Raw ingredient covariance PCA (203 dims)'),
    (axes[2], coords_std,  best_labels_std, recipes,   f1s,       f2s,
     'V10 - Standardized PCA (203 dims)  <- closest to reference'),
]

for ax, coords, labels, recs, f1v, f2v, title in specs:
    clrs = plt.cm.Set2(np.linspace(0, 1, max(len(set(labels)), 8)))
    for lbl in sorted(set(labels)):
        mask = labels == lbl
        pts  = coords[mask]
        c    = clrs[lbl % len(clrs)]
        ax.scatter(pts[:, 0], pts[:, 1], c=[c], s=120, alpha=0.85,
                   edgecolors='black', lw=0.5, zorder=3)
        if len(pts) > 1:
            cen = pts.mean(axis=0)
            std = pts.std(axis=0) + 1e-9
            el  = Ellipse(xy=cen, width=std[0]*4, height=std[1]*4,
                          edgecolor=c, fc='none', lw=1.4, ls='--', alpha=0.55)
            ax.add_patch(el)
    for i, rec in enumerate(recs):
        x, y     = coords[i]
        is_focus = rec in FOCUS_RECIPES
        ax.annotate(f'. {rec}', (x, y), xytext=(3, 3),
                    textcoords='offset points',
                    fontsize=7.5 if is_focus else 6,
                    fontweight='bold' if is_focus else 'normal',
                    color='red' if is_focus else '#1a1a2e')
        if is_focus:
            ax.scatter(x, y, marker='*', s=260, c='red', zorder=6,
                       edgecolors='darkred', lw=0.8)
    ax.axhline(0, color='gray', lw=0.4, ls='--', alpha=0.35)
    ax.axvline(0, color='gray', lw=0.4, ls='--', alpha=0.35)
    ax.set_xlabel(f'F1 ({f1v:.2f} %)', fontsize=10)
    ax.set_ylabel(f'F2 ({f2v:.2f} %)', fontsize=10)
    ax.set_title(f'{title}\nF1={f1v:.2f}%  F2={f2v:.2f}%', fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.15)

plt.suptitle('Three-Way Comparison  |  * = 185.028 & 188.740P  |  reference: F1=13.91%, F2=12.19%',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v10_threeway_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> v10_threeway_comparison.png')

In [ ]:
print('=' * 65)
print('FINAL VERDICT')
print('=' * 65)
print()
print(f'Reference           : F1=13.91%  F2=12.19%')
print(f'V9  (OT-agg)        : F1={ve_v9[0]:.2f}%  F2={ve_v9[1]:.2f}%  -- too few dimensions')
print(f'V10 (raw cov PCA)   : F1={f1_pct:.2f}%  F2={f2_pct:.2f}%  -- F1 dominated by outlier ingredient')
print(f'V10 (standardized)  : F1={f1s:.2f}%  F2={f2s:.2f}%  -- closest match')
print()
print('Focus recipes (standardized PCA):')
for rec in FOCUS_RECIPES:
    if rec not in recipes:
        print(f'  {rec}: not in dataset')
        continue
    idx = recipes.index(rec)
    pos = coords_std[idx]
    lbl = best_labels_std[idx]
    print(f'  {rec}: F1={pos[0]:+.3f}  F2={pos[1]:+.3f}  cluster={lbl}')
print()
print('Cluster assignments (standardized):')
for l in sorted(set(best_labels_std)):
    recs = [r for r, lb in zip(recipes, best_labels_std) if lb == l]
    flag = ' *' if any(r in FOCUS_RECIPES for r in recs) else ''
    print(f'  Cluster {l}{flag}: {recs}')